# Account Network Visualization

This notebook fetches network-analysis data for an account from the backend endpoint `/api/v1/lakehouse/network-analysis/account/{accountId}` and visualizes it as an interactive Plotly network graph. It uses the fallback account ID `cdtrAcct_3a1f3d24fb2046f2a28dc1fa506d6d69` when no account is supplied or when the requested account returns 404.

In [ ]:
# Imports and configuration
import os
source
def fetch_account_network(account_id, backend_url):
    url = f\

    params = {'tenantId': 'DEFAULT'}
    # Read auth token from environment if provided
    token = os.getenv('CASE_MGMT_API_TOKEN') or os.getenv('AUTH_TOKEN')
    headers = {'Authorization': f'Bearer {token}'} if token else None
    try:
        resp = requests.get(url, params=params, headers=headers, timeout=15)
    except Exception as e:
        print('Request error:', e)
        return None

    # If not found and we're not already using the fallback, try fallback
    if resp.status_code == 404 and account_id != FALLBACK_ACCOUNT_ID:
        print(f'Network data not found for {account_id}, trying fallback: {FALLBACK_ACCOUNT_ID}')
        url = f\

        try:
            resp = requests.get(url, params=params, headers=headers, timeout=15)
        except Exception as e:
            print('Request error (fallback):', e)
            return None

    # Handle auth errors explicitly
    if resp.status_code == 401:
        print('HTTP 401 Unauthorized: check CASE_MGMT_API_TOKEN or authentication with backend')
        try:
            resp.raise_for_status()
        except Exception as e:
            return None

    try:
        resp.raise_for_status()
    except Exception as e:
        print('HTTP error:', e)
        return None

    return resp.json()

# Example: fetch data now (will use CASE_MGMT_API_TOKEN if set)
data = fetch_account_network(accountId, backendUrl)
if not data:
    display(HTML(f\
else:
    print('Fetched network data keys:', list(data.keys()))

In [ ]:
# Normalise expected network payloads and build node/edge lists
# Expected formats (robust handling):
# { 'nodes': [{ 'id': 'acct1', 'label': 'Account 1', 'type': 'account', 'status': 'normal', 'isCenter': True }, ...],
#   'edges': [{ 'source': 'acct1', 'target': 'acct2', 'type': 'outbound', 'weight': 3 }, ...] }

nodes = []
edges = []

if data:
    if isinstance(data, dict):
        nodes = data.get('nodes') or data.get('vertices') or []
        edges = data.get('edges') or data.get('links') or []

# Ensure simple structures
nodes = [n for n in nodes if isinstance(n, dict)]
edges = [e for e in edges if isinstance(e, dict)]

print(f'Nodes: {len(nodes)}, Edges: {len(edges)}')

In [ ]:
# Build a simple layout (circular + center emphasis) and render with Plotly
def build_positions(nodes):
    positions = {}
    n = max(len(nodes), 1)
    # find center node (isCenter flag or the fallback account id)
    center_idx = 0
    for i, node in enumerate(nodes):
        if node.get('isCenter') or node.get('id') == accountId or node.get('id') == FALLBACK_ACCOUNT_ID:
            center_idx = i
            break

    # Place center at origin, others on circle
    radius = 300
    angle_step = 2 * math.pi / max(n - 1, 1)
    j = 0
    for i, node in enumerate(nodes):
        if i == center_idx:
            positions[node['id']] = (0.0, 0.0)
        else:
            angle = angle_step * j
            positions[node['id']] = (radius * math.cos(angle), radius * math.sin(angle))
            j += 1
    return positions

positions = build_positions(nodes)

# Create Plotly traces for edges (lines) and nodes (scatter)
edge_x = []
edge_y = []
for e in edges:
    s = e.get('source') or e.get('from')
    t = e.get('target') or e.get('to')
    if s in positions and t in positions:
        x0, y0 = positions[s]
        x1, y1 = positions[t]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]

edge_trace = go.Scatter(x=edge_x, y=edge_y, mode='lines', line=dict(width=1, color='#888'), hoverinfo='none')

# Node scatter - map types/status to marker styles
node_x = []
node_y = []
node_text = []
marker_size = []
marker_color = []

def color_for(node):
    # Prioritise status (alert/investigation) then type
    status = node.get('status', 'normal')
    t = node.get('type', 'account')
    if status in ('alert', 'flagged'):
        return '#ef4444'
    if status == 'investigation':
        return '#f59e0b'
    if t == 'counterparty':
        return '#6366f1'
    return '#60a5fa'

for n in nodes:
    nid = n.get('id')
    if nid not in positions:
        continue
    x, y = positions[nid]
    node_x.append(x)
    node_y.append(y)
    label = n.get('label') or nid
    node_text.append(f"{label} ({nid})")
    marker_size.append(30 if n.get('isCenter') else 18)
    marker_color.append(color_for(n))

node_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode='markers+text',
    text=[t.split(' ')[0] for t in node_text],
    hovertext=node_text,
    hoverinfo='text',
    marker=dict(
        showscale=False,
        color=marker_color,
        size=marker_size,
        line_width=2,
        line_color='#ffffff'
    ),
    textposition='bottom center'
)

fig = go.Figure(data=[edge_trace, node_trace], layout=go.Layout(
    title=f'Account Network: {accountId}',
    showlegend=False,
    hovermode='closest',
    margin=dict(b=20,l=5,r=5,t=40),
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    height=650,
))

if nodes:
    fig.show()
else:
    display(HTML('<div>No nodes to display</div>'))

## Notes
- The notebook attempts to fetch live network-analysis data from the backend.
- To run locally, set `CASE_MGMT_BACKEND_URL` and optionally `ACCOUNT_ID`. If `ACCOUNT_ID` is not set or the backend returns 404, the fallback account ID is used.
- The visualization uses a simple circular layout with the center account placed at the origin. For large graphs you may want to replace the layout with `networkx.spring_layout` if `networkx` is available.